# Air-Quality & Climate Assistant: SFT with QLoRA
Base model **Qwen2.5-3B-Instruct** (switch `MODEL_ID` to the 1.5B version if the GPU is busy), 4-bit NF4 quantization, LoRA r=16, free Colab **T4 GPU**.

Setup: Runtime > Change runtime type > **T4 GPU**. Optional Colab secret `HF_TOKEN` (a **write** token) for step 9. Run top to bottom.

## 1. Install

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate gradio

## 2. Data
Clones the project data, or asks you to upload `air_climate.jsonl` and `heldout_test.jsonl`. Optionally mixes in an extra Hugging Face Q&A dataset (set `HF_EXTRA` and its column names).

In [ ]:
import os, shutil, json
REPO = "https://github.com/lisaluis/air-climate-ai"
if not (os.path.exists("air_climate.jsonl") and os.path.exists("heldout_test.jsonl")):
    !git clone -q {REPO} repo
    for f in ["air_climate.jsonl", "heldout_test.jsonl"]:
        if os.path.exists(f"repo/llm/{f}"): shutil.copy(f"repo/llm/{f}", f)
if not (os.path.exists("air_climate.jsonl") and os.path.exists("heldout_test.jsonl")):
    from google.colab import files; files.upload()

SYSTEM = ("You are an air-quality and climate assistant for India. Explain ENSO, PM2.5 and pollution meteorology "
          "in plain language using the facts you were trained on. If you are not sure, say so and point to IMD, CPCB or the original study.")
train_rows = [json.loads(l) for l in open("air_climate.jsonl") if l.strip()]
held = [json.loads(l)["messages"] for l in open("heldout_test.jsonl") if l.strip()]

# OPTIONAL: extra public Q&A from the Hugging Face Hub (check the licence and spot-check rows first)
HF_EXTRA, Q_COL, A_COL, N_EXTRA = None, "question", "answer", 500    # e.g. HF_EXTRA = "owner/dataset-name"
if HF_EXTRA:
    from datasets import load_dataset
    extra = load_dataset(HF_EXTRA, split="train").select(range(N_EXTRA))
    train_rows += [{"messages": [{"role":"system","content":SYSTEM}, {"role":"user","content":r[Q_COL]}, {"role":"assistant","content":r[A_COL]}]} for r in extra]
with open("train_mixed.jsonl", "w") as f:
    for r in train_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(train_rows), "train examples,", len(held), "held-out examples")

## 3. Load the base model in 4-bit and test it BEFORE training

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"     # or "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto", dtype=torch.float16)

def ask(m, q, system=SYSTEM):
    text = tok.apply_chat_template([{"role":"system","content":system},{"role":"user","content":q}],
                                   tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=160, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

questions = [m[1]["content"] for m in held[:5]]
print("===== BEFORE TRAINING =====")
for q in questions: print("\nQ:", q, "\nA:", ask(model, q))

## 4. Fine-tune with QLoRA (SFT)
LoRA r=16, alpha=32 on all attention and MLP projections; 4 epochs, lr 2e-4, effective batch 8. Trainable LoRA weights are kept in fp32 (needed on the T4's fp16 mode).

In [ ]:
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

ds = load_dataset("json", data_files="train_mixed.jsonl", split="train")
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
cfg = SFTConfig(output_dir="out", num_train_epochs=4, per_device_train_batch_size=2, gradient_accumulation_steps=4,
                learning_rate=2e-4, logging_steps=5, fp16=True, bf16=False, max_length=512,
                save_strategy="no", report_to="none")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora, processing_class=tok)
for p in trainer.model.parameters():
    if p.requires_grad: p.data = p.data.float()
trainer.train()

## 5. Test AFTER training (same questions)

In [ ]:
trainer.model.eval()
print("===== AFTER TRAINING =====")
for q in questions: print("\nQ:", q, "\nA:", ask(trainer.model, q))
for q in ["What is the PM2.5 level in Pune today?", "Why was Mumbai dirtier in 2022-23?"]:
    print("\nQ:", q, "\nA:", ask(trainer.model, q))

## 6. Measured evaluation on held-out questions
Held-out questions are phrasings never seen in training. Score = share of the numbers in the reference answer that the model reproduces; for 'not sure' references the model scores 1 only if it also says it is not sure; text-only references are skipped. The base model is scored with the LoRA adapter switched off (same weights, same questions).

In [ ]:
import re
def numbers(t): return set(re.findall(r"\d+", t.replace(",", "")))
def score(ref, gen):
    if "not sure" in ref.lower(): return 1.0 if "not sure" in gen.lower() else 0.0
    r = numbers(ref)
    return len(r & numbers(gen)) / len(r) if r else None
def evaluate(m, label):
    s = [score(h[2]["content"], ask(m, h[1]["content"])) for h in held]
    sc = [x for x in s if x is not None]
    print(f"{label:<11} mean {sum(sc)/len(sc):.2f} over {len(sc)} scored questions", [None if x is None else round(x,2) for x in s])
    return sum(sc)/len(sc)
trainer.model.eval()
with trainer.model.disable_adapter(): b = evaluate(trainer.model, "BASE")
t = evaluate(trainer.model, "FINE-TUNED")
print(f"\nHeld-out score: base {b:.2f} -> fine-tuned {t:.2f}")

## 7. Compute and memory footprint (screenshot this)

In [ ]:
!nvidia-smi

## 8. Save the adapter (Google Drive backup, no token needed)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
trainer.model.save_pretrained("/content/drive/MyDrive/air-climate-lora"); tok.save_pretrained("/content/drive/MyDrive/air-climate-lora")

## 9. Upload to Hugging Face (optional)
Needs a Colab secret `HF_TOKEN` of type **Write**. Change `REPO_ID` to your username.

In [ ]:
from huggingface_hub import HfApi, create_repo
from google.colab import userdata
REPO_ID = "lisaluis717/air-climate-lora"
trainer.model.save_pretrained("air-climate-lora"); tok.save_pretrained("air-climate-lora")
token = userdata.get("HF_TOKEN")
create_repo(REPO_ID, token=token, exist_ok=True)
HfApi(token=token).upload_folder(folder_path="air-climate-lora", repo_id=REPO_ID)
print("https://huggingface.co/" + REPO_ID)

## 10. Chat UI (Gradio)
Run last; the cell stays running while the UI is up. Screenshot it with the public link visible.

In [ ]:
import gradio as gr
def chat(message, history): return ask(trainer.model, message)
gr.ChatInterface(chat, title="Air Quality & Climate Assistant (India)",
    description="Qwen2.5 fine-tuned with QLoRA (4-bit) on ENSO, PM2.5 and climate-zone Q&A. Informational only.",
    examples=["What is ENSO?", "Why was Mumbai dirtier in winter 2022-23?", "Does El Nino always make Delhi's air worse?"]
).launch(share=True, debug=True)